In [3]:
import requests
import h5py
import numpy as np
import os
import time
import tempfile

API_KEY = os.environ["TNG_API_KEY"]
BASE_URL = "https://www.tng-project.org/api/"

SIMULATION = "TNG100-1"

# Snapshotovi
snapshots = [99, 50, 33, 25]

# Polja koja preuzimamo
fields = [
    "SubhaloMassType",
    "SubhaloSFR",
    "SubhaloGasMetallicitySfrWeighted"
]

# Folder za čuvanje podataka
os.makedirs("TNG_groupcat_data", exist_ok=True)


def download_field(snap, field, max_retries=6):

    url = f"{BASE_URL}{SIMULATION}/files/groupcat-{snap}/"

    headers = {"API-Key": API_KEY}
    params = {"Subhalo": field}

    for attempt in range(max_retries):

        temp_path = None

        try:

            print(
                f"\nSnapshot {snap} | {field} | "
                f"Pokušaj {attempt + 1}/{max_retries}"
            )

            # Preuzimanje sa timeout-om i streaming opcijom
            with requests.get(
                url,
                headers=headers,
                params=params,
                timeout=(30, 600),
                stream=True
            ) as response:

                response.raise_for_status()

                # Privremeno čuvanje na disku
                with tempfile.NamedTemporaryFile(
                    suffix=".hdf5",
                    delete=False,
                    dir="TNG_groupcat_data"
                ) as temp_file:

                    temp_path = temp_file.name

                    for chunk in response.iter_content(
                        chunk_size=8 * 1024 * 1024
                    ):
                        if chunk:
                            temp_file.write(chunk)

            # Čitanje HDF5 fajla
            with h5py.File(temp_path, "r") as f:

                if "Subhalo" in f and field in f["Subhalo"]:
                    data = f["Subhalo"][field][:]

                elif field in f:
                    data = f[field][:]

                else:
                    raise KeyError(
                        f"Polje {field} nije pronađeno. "
                        f"Dostupni ključevi: {list(f.keys())}"
                    )

            print("Uspešno preuzeto")
            print("Oblik niza:", data.shape)

            return data

        except (requests.RequestException, OSError) as e:

            print("\nGreška:", e)

            if attempt < max_retries - 1:

                wait = min(10 * (2 ** attempt), 120)

                print(
                    f"Ponavljam zahtev za {wait} sekundi..."
                )

                time.sleep(wait)

            else:

                raise RuntimeError(
                    f"Preuzimanje nije uspelo: "
                    f"snapshot {snap}, polje {field}"
                ) from e

        finally:

            # Brisanje privremenog fajla
            if temp_path is not None and os.path.exists(temp_path):
                os.remove(temp_path)


def download_snapshot(snap):

    print(f"\nPreuzimanje snapshot-a {snap}")
    print("=" * 50)

    data = {}

    # Preuzimanje sva tri polja
    for field in fields:

        print(f"\nPreuzimam: {field}")

        data[field] = download_field(snap, field)

    # Provera broja subhaloa
    lengths = [len(data[field]) for field in fields]

    if len(set(lengths)) != 1:
        raise ValueError(
            "Broj subhaloa nije isti za sva polja"
        )

    N = lengths[0]

    # ID subhaloa odgovara indeksu u katalogu
    data["SubhaloID"] = np.arange(N, dtype=np.int64)

    # Čuvanje podataka
    filename = f"TNG_groupcat_data/snapshot_{snap}.npz"

    np.savez_compressed(filename, **data)

    print("\n" + "=" * 50)
    print("Snapshot saved")
    print("Broj subhaloa:", N)
    print("Sačuvano u:", filename)

    return data


# Preuzimanje svih snapshotova
all_data = {}

for snap in snapshots:

    all_data[snap] = download_snapshot(snap)

print("\nSva preuyimanja gotova")


Preuzimanje snapshot-a 99

Preuzimam: SubhaloMassType

Snapshot 99 | SubhaloMassType | Pokušaj 1/6

Greška: 504 Server Error: Gateway Time-out for url: https://data-eu.tng-project.org/file/groupcat/L75n1820TNG/99/req.Subhalo.SubhaloMassType/?token=f772f824e08e8fa784ad
Ponavljam zahtev za 10 sekundi...

Snapshot 99 | SubhaloMassType | Pokušaj 2/6

Greška: 504 Server Error: Gateway Time-out for url: https://data-eu.tng-project.org/file/groupcat/L75n1820TNG/99/req.Subhalo.SubhaloMassType/?token=05db6e74a369a0e979ce
Ponavljam zahtev za 20 sekundi...

Snapshot 99 | SubhaloMassType | Pokušaj 3/6
Uspešno preuzeto
Oblik niza: (4371211, 6)

Preuzimam: SubhaloSFR

Snapshot 99 | SubhaloSFR | Pokušaj 1/6
Uspešno preuzeto
Oblik niza: (4371211,)

Preuzimam: SubhaloGasMetallicitySfrWeighted

Snapshot 99 | SubhaloGasMetallicitySfrWeighted | Pokušaj 1/6
Uspešno preuzeto
Oblik niza: (4371211,)

Snapshot saved
Broj subhaloa: 4371211
Sačuvano u: TNG_groupcat_data/snapshot_99.npz

Preuzimanje snapshot-a 5